# 엘리베이터 폭력 감지 — Colab 학습 노트북
MobileNetV2 + LSTM / TCN 학습 → Recall, Precision, F1, Accuracy, 속도 측정

**실행 전 체크**
1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. 아래 `REPO_URL`을 팀 repo 주소로 변경
3. 위에서부터 순서대로 실행 (자세한 설명: `docs/GUIDE.md`)

## 0. 환경 준비

In [ ]:
import os, sys
REPO_URL = "https://github.com/<팀>/elevator-violence-detection.git"   # ← 바꾸세요
REPO_DIR = "/content/elevator-violence-detection"

import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/rvd", exist_ok=True)   # 특징 파일 백업용

In [ ]:
# repo 가져오기 (private repo면 토큰 필요: 맨 아래 7번 셀 참고)
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q opencv-python-headless scikit-learn

## 1. 데이터 받기 (Kaggle SCVD)
Kaggle → Settings → API → **Create New Token** 으로 받은 `kaggle.json`을 업로드합니다.

In [ ]:
from google.colab import files
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    files.upload()   # kaggle.json 선택
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
DATA = "/content/scvd"
!kaggle datasets download -d toluwaniaremu/smartcity-cctv-violence-detection-dataset-scvd -p {DATA} --unzip

## 2. 폴더 구조와 라벨링 확인
출력된 구조를 `docs/datasets.md`에 기록하세요. 비폭력/폭력 개수가 맞는지 꼭 확인합니다.
폴더 이름 때문에 라벨이 틀리면 `--neg "키워드1,키워드2"` 를 추가하세요.

In [ ]:
!find {DATA} -maxdepth 3 -type d | head -30
!find {DATA} -type f \( -name "*.mp4" -o -name "*.avi" -o -name "*.mov" -o -name "*.mkv" \) | wc -l

In [ ]:
# 라벨링 확인만 (특징 추출 안 함)
!python rvd_pipeline.py extract --data {DATA} --out feats --dry-run

## 3. 특징 추출 (MobileNetV2) — 가장 오래 걸리는 단계
끝나면 Drive에 백업해서 런타임이 끊겨도 다시 하지 않게 합니다.

In [ ]:
!python rvd_pipeline.py extract --data {DATA} --out feats
!cp feats/feats.npz /content/drive/MyDrive/rvd/feats.npz && echo "Drive 백업 완료"

In [ ]:
# (런타임이 끊겼다면) Drive에서 복원
# os.makedirs("feats", exist_ok=True)
# !cp /content/drive/MyDrive/rvd/feats.npz feats/feats.npz

## 4. LSTM / TCN 학습 + 지표

In [ ]:
!python rvd_pipeline.py train --feats feats --out results --epochs 50

In [ ]:
import json, pandas as pd
m = json.load(open("results/metrics.json", encoding="utf-8"))
df = pd.DataFrame(m).T[["recall","precision","f1","accuracy","TP","FP","FN","TN","params"]]
display(df.round(4))

## 5. 속도 측정 (Colab = 참고용)
Colab 수치는 **참고용**입니다. 최종 FPS/추론시간은 실제 배포 기기에서 `speed` 단계를 다시 실행하세요.

In [ ]:
!python rvd_pipeline.py speed --data {DATA} --out results --n 30

In [ ]:
s = json.load(open("results/speed.json", encoding="utf-8"))
env = s.pop("_env")
print("측정 환경:", env)
display(pd.DataFrame(s).T[["preprocess_ms","backbone_ms","temporal_ms","inference_ms_median","inference_ms_p95","fps"]].round(2))

## 6. 결과 백업
`docs/experiments.md` 표에 위 결과를 옮겨 적으세요 (날짜, 모델, 설정, 지표, 메모).

In [ ]:
!mkdir -p /content/drive/MyDrive/rvd/results && cp results/*.json /content/drive/MyDrive/rvd/results/ && echo "결과 json Drive 백업 완료"

## 7. GitHub에 기록 (push)
**준비**: GitHub → Settings → Developer settings → Fine-grained token (해당 repo, Contents: Read and write) 생성 →
Colab 왼쪽 🔑 **보안 비밀** 에 이름 `GITHUB_TOKEN`으로 저장 + 노트북 액세스 켜기.

> 토큰을 코드에 직접 붙여넣지 마세요. 노트북이 repo에 올라가면 토큰이 공개됩니다.

In [ ]:
from google.colab import userdata
BRANCH = "exp/01-baseline"                       # ← 실험마다 변경
MSG    = "eval: baseline 결과(metrics/speed) 추가"  # ← 커밋 메시지
USER, EMAIL = "your-name", "your-email@example.com"   # ← 본인 정보

token = userdata.get("GITHUB_TOKEN")
repo_path = REPO_URL.replace("https://", "")
os.environ["GH_AUTH_URL"] = f"https://{token}@{repo_path}"

!git config user.name "{USER}"
!git config user.email "{EMAIL}"
!git checkout -B {BRANCH}
!git add results/*.json docs/ notebooks/
!git commit -m "{MSG}" || echo "커밋할 변경 없음"
!git push "$GH_AUTH_URL" {BRANCH}

push 후 GitHub에서 **Pull Request**를 만들어 팀원 확인 → `main`에 merge 하세요.